# All-28 Jev-CeNN · Forced attention replacement → COMPLETE Hugging Face model (V5)

**Purpose:** Publish your accepted-for-research **all-28 forced attention** architecture, retaining the *original exact FFN weights*, both layer normalizations, residual pathways, 28 encoder layers, and typed-decision head. **There is no quality, teacher-agreement, cosine, or speed acceptance gate.** All 28 attention replacements are installed unconditionally.

**Fastest path — use your completed V4 model:**
1. In the same runtime, keep `/content/Jev_CeNN_All28_AttentionOnly_BruteForce_V4/` available, **or** upload the previously downloaded `all28_results.zip` using Colab's Files sidebar to `/content/all28_results.zip` (set `V4_ZIP_PATH` to its path).
2. Run all cells. They reconstruct a FULL model from the 28 `forced_best_attention_L00.pt` ... `L27.pt` files, export full `model.safetensors`, self-contained config/tokenizer, documented `all28_hf.py` reloader, and publish automatically to your authenticated Hugging Face namespace. Reusing all checkpoints does **not** train again.
3. If no checkpoints are available and `TRAIN_IF_MISSING=True`, Colab trains missing attention layers from scratch using the V4 search; **nothing is rejected**. This can be expensive, so reuse the V4 ZIP to reproduce the 94.8%-agreement research model.

**Privacy:** Hugging Face model repository is **private by default**; set `HF_PRIVATE=False` to publish publicly after checking the original model's license and distribution rights. Tokens are obtained from the Colab/HF authentication flow, never written into the model folder.

**Outputs:** A complete runnable Hugging Face model repo containing native FFNs and all 28 custom attention modules, a loader and usage example, plus a JSON manifest, full weights, source config and tokenizer. The notebook performs an independent reload+logit round-trip test before upload. The source score of 94.8% agreement applies to your **previous V4 run**, not automatically to a fresh retraining.

**Requirements:** A GPU is recommended for training missing checkpoints; exporting the existing ZIP can be done on CPU, subject to sufficient RAM/disk. No original checkpoint download is needed after publishing the complete model.


In [ ]:
#@title 1. Setup, local V4 checkpoint reuse, and Hugging Face destination
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@eba220d8131dbdf06f748545a21831327e5ad37d"

from pathlib import Path
import copy, gc, hashlib, json, math, random, shutil, sys, time, zipfile
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download, HfApi, get_token, login
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question, full_attention_indices
)
from tinycenn_lm.decision_training import require_finite
from tinycenn_lm.jev_cenn_full_local_repeat import cpu_state_snapshot
from tinycenn_lm.all28_hf import WindowCeNNAttention, build_all28_from_dir
import tinycenn_lm.all28_hf as exported_architecture

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
V4_CHECKPOINT_DIR="/content/Jev_CeNN_All28_AttentionOnly_BruteForce_V4" #@param {type:"string"}
V4_ZIP_PATH="/content/all28_results.zip" #@param {type:"string"}
TRAIN_IF_MISSING=True #@param {type:"boolean"}
HF_REPO_NAME="jev-cenn-all28-attention-frozen-ffn-v5" #@param {type:"string"}
HF_PRIVATE=True #@param {type:"boolean"}
UPLOAD_TO_HF=True #@param {type:"boolean"}
OUTPUT_DIR=Path("/content/Jev_CeNN_All28_HF_Export_V5")
CHECKPOINT_DIR=OUTPUT_DIR/"per_layer_checkpoints"
EXPORT_DIR=OUTPUT_DIR/"huggingface_model"
CHECKPOINT_DIR.mkdir(parents=True,exist_ok=True)
EXPORT_DIR.mkdir(parents=True,exist_ok=True)

LAYER_ORDER=tuple(range(28))
GLOBAL_GRID=((129,"anchors",16),(129,"anchors",32),
 (257,"anchors",16),(257,"anchors",32),(385,"anchors",16),
 (385,"anchors",32),(449,"anchors",16),(481,"anchors",16))
LOCAL_GRID=((33,"local",0),(65,"local",0),(97,"local",0))
TOP_K_TRAIN=2
QUICK_PROBE_BATCHES=3
TRAIN_STEPS_GLOBAL=120
TRAIN_STEPS_LOCAL=80
CACHE_BATCHES_GLOBAL=80
CACHE_BATCHES_LOCAL=48
PROBE_BATCHES=12
BATCH=2
PROBE_EVERY=30
LR_ATTENTION=1.5e-5
LR_CENN=2e-4
COSINE_WEIGHT=3.0
TRAIN_CASES=700
DEV_CASES=220
HOLDOUT_CASES=200
PROBE_CASES=80
DEV_GATE_DECISIONS=240
HOLDOUT_DECISIONS=500
MAX_LEN=512

random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)

# Read ONLY correctly-named .pt files, never untrusted ZIP paths.
def checkpoint_name(i):
    return f"forced_best_attention_L{i:02d}.pt"

def find_local_checkpoint(i):
    search_dirs=(CHECKPOINT_DIR,Path(V4_CHECKPOINT_DIR),OUTPUT_DIR,
                 Path("/content"))
    for root in search_dirs:
        p=root/checkpoint_name(i)
        if p.is_file():return p
    return None

archive=Path(V4_ZIP_PATH)
if archive.is_file():
    with zipfile.ZipFile(archive) as zf:
        by_name={Path(n).name:n for n in zf.namelist()
                 if Path(n).name.startswith("forced_best_attention_L")
                 and Path(n).name.endswith(".pt")}
        for i in LAYER_ORDER:
            if find_local_checkpoint(i) is not None:continue
            n=by_name.get(checkpoint_name(i))
            if n is not None:
                (CHECKPOINT_DIR/checkpoint_name(i)).write_bytes(zf.read(n))
    print("Imported previous V4 checkpoints from:",archive)

missing_layers=[i for i in LAYER_ORDER if find_local_checkpoint(i) is None]
NEED_TRAINING=bool(missing_layers)
print(f"Found {28-len(missing_layers)}/28 V4 attention checkpoints.")
print("Missing:",missing_layers)
if NEED_TRAINING and not TRAIN_IF_MISSING:
    raise RuntimeError("Some V4 checkpoints are missing. Upload all28_results.zip "
                       "or set TRAIN_IF_MISSING=True to train missing layers.")
if NEED_TRAINING and not torch.cuda.is_available():
    raise RuntimeError("Missing attention checkpoints require a GPU. Upload "
                       "the completed V4 ZIP to publish without retraining.")
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
      "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"]))
student_eval,source_cfg=build_checkpoint_model(source_dir)
student_eval.eval().requires_grad_(False)
student_eval.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
teacher=None
GLOBAL_LAYERS=tuple(full_attention_indices(student_eval))
assert len(student_eval.encoder.layers)==28
assert GLOBAL_LAYERS==tuple(range(0,28,3)),GLOBAL_LAYERS
print("Forced architecture: all 28 ATTENTION modules, original FFNs preserved.")


In [ ]:
#@title 2. Dataset partitions (ONLY when missing checkpoints require new training)
if NEED_TRAINING:
    teacher,_=build_checkpoint_model(source_dir)
    teacher.to(device).eval().requires_grad_(False)
    teacher.encoder.config.reference_compile=False
    student_eval.to(device)
    #@title 3. Partition dataset by CASE: TRAIN, PROBE, DEV-GATE and sealed HOLDOUT
    def _jsonish(x):
        if isinstance(x, str):
            try:
                return json.loads(x)
            except Exception:
                return x
        return x

    def _canon_label(x):
        s = str(x).strip().lower()
        try:
            v = float(s)
            if math.isfinite(v) and v.is_integer():
                return str(int(v))
        except Exception:
            pass
        return s

    def _gold_metadata(q, gold_answer):
        if not isinstance(gold_answer, dict):
            raise ValueError("missing gold answer")
        probs_raw = _jsonish(gold_answer.get("probabilities", {}))
        if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
            raise ValueError("gold probabilities missing or malformed")

        gold_keys = [str(k) for k in probs_raw.keys()]
        if q["t"] == "choice":
            labels = [str(k) for k in q["crit"].keys()]
        elif q["t"] == "noul":
            labels = ["false", "true"]
        else:
            labels = list(gold_keys)
            if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
                labels = sorted(labels, key=lambda k: int(_canon_label(k)))

        lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
        if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
            raise ValueError("Gold labels do not match encoded option order")

        gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
        gold_probs = np.clip(gold_probs, 0.0, None)
        if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
            raise ValueError("invalid gold probability vector")
        gold_probs /= gold_probs.sum()

        gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
        canon_labels = [_canon_label(k) for k in labels]
        if gold_label not in canon_labels:
            raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
        return labels, gold_probs.tolist(), canon_labels.index(gold_label)

    def make_items_from_rows(rows, seed, shuffle_items):
        out = []
        head_max_len = int(source_cfg.get("head_max_len", 192))
        for row in rows:
            state = _jsonish(row["state"])
            questions = _jsonish(row["questions"])
            gold = _jsonish(row["gold"])
            case_id = str(row.get("id", ""))
            workflow = str(row.get("workflow", "unknown"))
            state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
            state_ids = tokenizer(
                state_text.replace(tokenizer.mask_token, " "),
                add_special_tokens=False,
            )["input_ids"]

            for qid, qdef in questions.items():
                q = normalize_question(qdef)
                ids, markers = build_sequence(
                    tokenizer,
                    state,
                    q,
                    max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                    head_max_len=head_max_len,
                    truncate_left=isinstance(state, list),
                    state_ids=state_ids,
                )
                if len(markers) < 2:
                    continue
                labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
                if len(labels) != len(markers):
                    raise ValueError(
                        f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                    )
                out.append({
                    "ids": ids,
                    "markers": markers,
                    "qtype": QTYPES[q["t"]],
                    "qtype_name": q["t"],
                    "qid": str(qid),
                    "case_id": case_id,
                    "workflow": workflow,
                    "labels": labels,
                    "gold_probs": gold_probs,
                    "gold_index": int(gold_index),
                })
        if shuffle_items:
            random.Random(seed).shuffle(out)
        return out

    from tinycenn_lm.jev_cenn_selective import (
        verify_case_partition, judge_replacement, plan_active_layers,
    )
    rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
    rng=random.Random(SEED)
    rng.shuffle(rows)
    assert TRAIN_CASES+DEV_CASES+HOLDOUT_CASES<=len(rows), (
        f"Required TRAIN+DEV+HOLDOUT cases exceeds {len(rows)} dataset TRAIN cases")
    train_rows=rows[:TRAIN_CASES]
    dev_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES]
    holdout_rows=rows[TRAIN_CASES+DEV_CASES:TRAIN_CASES+DEV_CASES+HOLDOUT_CASES]
    split_info=verify_case_partition(train_rows,dev_rows,holdout_rows)
    probe_rows=dev_rows[:PROBE_CASES]
    gate_rows=dev_rows[PROBE_CASES:]
    verify_case_partition(train_rows,probe_rows,gate_rows)
    assert not {str(x["id"]) for x in holdout_rows}&{
        str(x["id"]) for x in dev_rows}
    train_items=make_items_from_rows(train_rows,SEED,True)
    probe_items=make_items_from_rows(probe_rows,SEED+1,True)
    gate_items=make_items_from_rows(gate_rows,SEED+2,True)
    # IMPORTANT: holdout_items are NOT constructed yet. The final evaluation
    # cell materializes these previously sealed rows after all decisions are made.
    assert len(gate_items)>=DEV_GATE_DECISIONS
    assert len(probe_items)>=PROBE_BATCHES*BATCH
    print("Independent partitions:",split_info,
          "probe cases",len(probe_rows),"gate cases",len(gate_rows))
    print("Optimizer/probe/gate decisions:",len(train_items),
          len(probe_items),len(gate_items))

    def direct_batch(selected):
        b = collate_items(selected, int(tokenizer.pad_token_id))
        return {k: v.to(device) for k, v in b.items()}

    def batch(items, offset, n=BATCH):
        selected = [items[(offset + i) % len(items)] for i in range(n)]
        return direct_batch(selected)

    # Only case-isolated TRAIN / PROBE / DEV-GATE inputs are available here.
    # The independent HOLDOUT remains sealed until the final evaluation cell.
    print("TRAIN / PROBE / DEV-GATE decisions:",
          len(train_items), len(probe_items), len(gate_items))
    print("Official TEST untouched; independent HOLDOUT not materialized.")

else:
    print("All 28 checkpoints available — dataset download/training skipped.")


In [ ]:
#@title 3. Cache current prefix targets only if training is needed
if NEED_TRAINING:
    #@title 3. Correct prefix caching: student-native targets for cumulative distillation
    def check_finite(tensor,caption):
        if not torch.isfinite(tensor.float()).all():
            raise FloatingPointError(caption)

    def to_cuda(row):
        return {k:v.to(device,non_blocking=True) for k,v in row.items()}

    def frozen_fingerprint(model):
        sha=hashlib.sha256()
        for i,layer in enumerate(model.encoder.layers):
            for module_name in ("mlp","mlp_norm","attn_norm"):
                module=getattr(layer,module_name)
                for key,tensor in module.state_dict().items():
                    sha.update(f"{i}:{module_name}:{key}".encode())
                    sha.update(tensor.detach().cpu().contiguous().numpy().tobytes())
        return sha.hexdigest()

    FFN_NORM_FINGERPRINT=frozen_fingerprint(student_eval)
    def assert_frozen_ffn(model=student_eval):
        actual=frozen_fingerprint(model)
        assert actual==FFN_NORM_FINGERPRINT,(
            "Pretrained FFN or norm weights changed! Model invalid.")
        for layer in model.encoder.layers:
            assert not any(x.requires_grad for x in layer.mlp.parameters())
            assert not any(x.requires_grad for x in layer.mlp_norm.parameters())
            assert not any(x.requires_grad for x in layer.attn_norm.parameters())

    def cache_layer(layer_index,items,n_batches,seed):
        chosen=random.Random(seed).sample(items,min(len(items),n_batches*BATCH))
        out=[]
        # Native attention must still be installed at target at cache time.
        assert not isinstance(student_eval.encoder.layers[layer_index].attn,WindowCeNNAttention)
        for at in range(0,len(chosen)-BATCH+1,BATCH):
            selected=chosen[at:at+BATCH]
            inputs=direct_batch(selected)
            teach={};current={}
            def before_teacher(mod,args,kwargs):
                teach["x"]=args[0].detach()
                teach["position_ids"]=kwargs.get("position_ids")
            def after_teacher(mod,args,kwargs,result):
                teach["target"]=result[0].detach()
            def before_student(mod,args,kwargs):
                current["x"]=args[0].detach()
                current["position_ids"]=kwargs.get("position_ids")
            def after_student(mod,args,kwargs,result):
                current["native_target"]=result[0].detach()
            tl=teacher.encoder.layers[layer_index]
            sl=student_eval.encoder.layers[layer_index]
            h1=tl.register_forward_pre_hook(before_teacher,with_kwargs=True)
            h2=tl.register_forward_hook(after_teacher,with_kwargs=True)
            try:
                with torch.inference_mode():teacher(**inputs)
            finally:h1.remove();h2.remove()
            h3=sl.register_forward_pre_hook(before_student,with_kwargs=True)
            h4=sl.register_forward_hook(after_student,with_kwargs=True)
            try:
                with torch.inference_mode():student_eval(**inputs)
            finally:h3.remove();h4.remove()
            assert all(k in teach for k in ("x","position_ids","target"))
            assert all(k in current for k in ("x","position_ids","native_target"))
            row={
                "x":teach["x"],
                "teacher_target":teach["target"],
                "rollout":current["x"],
                "native_target":current["native_target"],
                "valid":inputs["attention_mask"].bool(),
                "position_ids":current["position_ids"],
            }
            out.append({
                k:(v.detach().float().cpu().contiguous()
                   if v.is_floating_point() else v.detach().cpu().contiguous())
                for k,v in row.items()})
        return out

    def new_layer_caches(layer_index):
        n=CACHE_BATCHES_GLOBAL if layer_index in GLOBAL_LAYERS else CACHE_BATCHES_LOCAL
        train=cache_layer(layer_index,train_items,n,SEED+layer_index*13+1)
        probe=cache_layer(layer_index,probe_items,PROBE_BATCHES,SEED+layer_index*13+2)
        return train,probe

    print("SHA-256 of all 28 FFNs + attention/FFN LayerNorms:",
          FFN_NORM_FINGERPRINT)

else:
    print("No activation caching required for the existing V4 checkpoints.")


In [ ]:
#@title 4. Model trial functions, used only for missing layer training
if NEED_TRAINING:
    def make_candidate(layer_idx,config):
        window,pattern,anchors=config
        return WindowCeNNAttention(
            teacher.encoder.layers[layer_idx].attn,
            window,pattern,anchors).to(device)

    @torch.inference_mode()
    def feature_probe(candidate,layer_idx,rows):
        """Incremental fidelity against the native target on CURRENT student inputs."""
        layer=student_eval.encoder.layers[layer_idx]
        native=layer.attn
        layer.attn=candidate
        ce=err=energy=tokens=0.0
        candidate.eval()
        try:
            for row in rows:
                t=to_cuda(row)
                out=layer(t["rollout"],attention_mask=t["valid"],
                          position_ids=t["position_ids"])[0]
                p=out.float()[t["valid"]]
                ref=t["native_target"].float()[t["valid"]]
                ce+=float(F.cosine_similarity(p,ref,dim=-1).sum())
                err+=float((p-ref).square().sum())
                energy+=float(ref.square().sum())
                tokens+=int(t["valid"].sum())
            return {"cosine":ce/max(tokens,1),"nmse":err/max(energy,1e-9),
                    "tokens":int(tokens)}
        finally:layer.attn=native

    def local_loss(out,target,valid):
        p=out.float()[valid];ref=target.float()[valid]
        nmse=(p-ref).square().sum()/ref.square().sum().clamp_min(1e-8)
        cos=F.cosine_similarity(p,ref,dim=-1).mean()
        return nmse+COSINE_WEIGHT*(1-cos)

    @torch.inference_mode()
    def verify_native_attention(layer_idx,rows):
        # Native-equivalent full/local control must reproduce same current-student outputs.
        original=student_eval.encoder.layers[layer_idx].attn
        config=((MAX_LEN,"full",0) if layer_idx in GLOBAL_LAYERS
                else (129,"local",0))
        replica=make_candidate(layer_idx,config)
        replica.use_cenn=False
        q=feature_probe(replica,layer_idx,rows[:2])
        del replica
        if q["cosine"]<0.9999 or q["nmse"]>0.00005:
            raise RuntimeError(f"Layer {layer_idx}: native attention not reproduced: {q}")
        return q

    print("Per-layer attention implementation ready; native FFNs untouched.")

    #@title 6. Brute-force masked search, train top candidates, CeNN on/off
    @torch.inference_mode()
    def zero_shot_scan(layer_idx,configs,probe_rows):
        scanned=[]
        for cfg in configs:
            candidate=make_candidate(layer_idx,cfg).eval()
            candidate.use_cenn=False
            q=feature_probe(candidate,layer_idx,probe_rows[:QUICK_PROBE_BATCHES])
            score=q["nmse"]+COSINE_WEIGHT*(1-q["cosine"])
            scanned.append({"config":list(cfg),"score":score,"local":q})
            del candidate
        scanned.sort(key=lambda x:x["score"])
        return scanned

    def train_attention(candidate,layer_idx,train_cache,probe_cache,steps):
        candidate.backend="dense"
        candidate.train().requires_grad_(True)
        candidate.rotary_emb.requires_grad_(False)
        candidate.out_drop.requires_grad_(False)
        params_qkv=list(candidate.Wqkv.parameters())+list(candidate.Wo.parameters())
        params_ce=list(candidate.cenn.parameters())+[candidate.cenn_gain]
        opt=torch.optim.AdamW([
            {"params":params_qkv,"lr":LR_ATTENTION},
            {"params":params_ce,"lr":LR_CENN}],weight_decay=1e-4)
        best_rank=float("inf");best=None;best_step=0
        history=[];indices=list(range(len(train_cache)));index=0
        rng=random.Random(SEED+layer_idx*719+candidate.window)
        layer=student_eval.encoder.layers[layer_idx]
        native=layer.attn
        for step in range(1,steps+1):
            if index==0:rng.shuffle(indices)
            record=to_cuda(train_cache[indices[index]])
            index=(index+1)%len(indices)
            candidate.train()
            opt.zero_grad(set_to_none=True)
            layer.attn=candidate
            try:
                output=layer(record["rollout"],attention_mask=record["valid"],
                             position_ids=record["position_ids"])[0]
                loss=local_loss(output,record["native_target"],record["valid"])
                require_finite(loss,"attention-only loss")
                loss.backward()
            finally:
                layer.attn=native
            torch.nn.utils.clip_grad_norm_(params_qkv+params_ce,0.5)
            opt.step()
            if step==1 or step%PROBE_EVERY==0 or step==steps:
                cand_q=feature_probe(candidate,layer_idx,probe_cache)
                rank=cand_q["nmse"]+COSINE_WEIGHT*(1-cand_q["cosine"])
                history.append({"step":step,"probe":cand_q,"score":rank})
                if rank<best_rank:
                    best_rank,best_step,best=rank,step,cpu_state_snapshot(candidate)
                print(f"  L{layer_idx:02d} W{candidate.window} "
                      f"{candidate.pattern} train {step}/{steps} "
                      f"cos={cand_q['cosine']:.6f} nmse={cand_q['nmse']:.5g}")
        assert best is not None
        candidate.load_state_dict(best,strict=True)
        candidate.eval().requires_grad_(False)
        del opt,best
        return {"best_step":best_step,"steps":steps,"history":history}


else:
    print("Reusing previous 28 trained attention modules without optimizer updates.")


In [ ]:
#@title 5. Train ONLY missing layers; always save best attention, never reject
if NEED_TRAINING:
    GATE_ITEMS=gate_items[:DEV_GATE_DECISIONS]
    GATE_CACHE=cache_teacher_predictions(GATE_ITEMS)
    print("Training missing checkpoints:",missing_layers)
    for layer_idx in missing_layers:
        cache_train,cache_probe=new_layer_caches(layer_idx)
        reference=verify_native_attention(layer_idx,cache_probe)
        grid=GLOBAL_GRID if layer_idx in GLOBAL_LAYERS else LOCAL_GRID
        scan=zero_shot_scan(layer_idx,grid,cache_probe)
        candidates=scan[:TOP_K_TRAIN]
        best=None;best_rank=None
        for shortlist in candidates:
            cfg=tuple(shortlist["config"])
            candidate=make_candidate(layer_idx,cfg)
            nsteps=TRAIN_STEPS_GLOBAL if layer_idx in GLOBAL_LAYERS else TRAIN_STEPS_LOCAL
            fit=train_attention(candidate,layer_idx,cache_train,cache_probe,nsteps)
            for ce in (False,True):
                candidate.use_cenn=ce
                probe=feature_probe(candidate,layer_idx,cache_probe)
                # Do NOT gate anything: simply select best local fidelity.
                rank=(probe["cosine"],-probe["nmse"])
                if best_rank is None or rank>best_rank:
                    best_rank=rank
                    best=(cfg,ce,cpu_state_snapshot(candidate),probe)
            del candidate
        assert best is not None
        config,cenn,weights,probe=best
        output=CHECKPOINT_DIR/checkpoint_name(layer_idx)
        torch.save({"layer":layer_idx,"config":list(config),
                    "cenn_on":cenn,"state":weights,
                    "selection_probe":probe,"quality_gate_not_required":True},
                   output)
        print("Saved FORCED",layer_idx,config,"CeNN",cenn,
              "cosine",round(probe["cosine"],6))
        del best,cache_train,cache_probe
        gc.collect()
        torch.cuda.empty_cache()
    del teacher
    teacher=None
    student_eval.cpu()
    gc.collect();torch.cuda.empty_cache()

assert all(find_local_checkpoint(i) is not None for i in LAYER_ORDER)
print("All 28 attention checkpoints ready — no quality gate.")


In [ ]:
#@title 6. Unconditionally replace ALL 28 attentions; verify original FFNs byte-for-byte
def pretrained_sha_for_frozen_parts(model):
    sha=hashlib.sha256()
    for idx,layer in enumerate(model.encoder.layers):
        for part in ("mlp","mlp_norm","attn_norm"):
            for key,tensor in getattr(layer,part).state_dict().items():
                sha.update(f"{idx}/{part}/{key}".encode())
                sha.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return sha.hexdigest()

student_eval.cpu().eval().requires_grad_(False)
before_ffn_sha=pretrained_sha_for_frozen_parts(student_eval)
manifest_layers=[]
for i in LAYER_ORDER:
    saved=torch.load(find_local_checkpoint(i),map_location="cpu",weights_only=True)
    assert int(saved["layer"])==i
    assert bool(saved.get("quality_gate_not_required",True))
    config=list(saved["config"])
    assert len(config)==3 and config[1] in ("local","anchors","full")
    native=student_eval.encoder.layers[i].attn
    assert not isinstance(native,WindowCeNNAttention)
    converted=WindowCeNNAttention(native,int(config[0]),str(config[1]),int(config[2]))
    converted.load_state_dict(saved["state"],strict=True)
    converted.use_cenn=bool(saved["cenn_on"])
    converted.eval().requires_grad_(False)
    student_eval.encoder.layers[i].attn=converted
    manifest_layers.append({"layer":i,"window":int(config[0]),
        "pattern":str(config[1]),"anchors":int(config[2]),
        "cenn_on":converted.use_cenn})
    print(f"L{i:02d}: {config} CeNN={converted.use_cenn} FORCED-INSTALLED")
    del saved
after_ffn_sha=pretrained_sha_for_frozen_parts(student_eval)
assert before_ffn_sha==after_ffn_sha,"FFNs or LayerNorms were changed!"
assert len(manifest_layers)==28
assert all(isinstance(layer.attn,WindowCeNNAttention)
           for layer in student_eval.encoder.layers)
assert all(not x.requires_grad for x in student_eval.parameters())
print("ALL 28 ATTENTION MODULES REPLACED.")
print("All 28 original FFNs and normalizations: BIT-IDENTICAL SHA256",after_ffn_sha)


In [ ]:
#@title 7. Export FULL safetensors model + native configs/tokenizer and HF loader
from safetensors.torch import save_model
import datetime
# Model has ALL layers, not just patches. No teacher download needed by loader.
student_eval.eval().cpu()
student_eval.encoder.config.reference_compile=False
model_file=EXPORT_DIR/"model.safetensors"
save_model(student_eval,str(model_file))
assert model_file.is_file() and model_file.stat().st_size>0
tok_dir=EXPORT_DIR/"tokenizer"
tokenizer.save_pretrained(str(tok_dir))
enc_dir=EXPORT_DIR/"encoder"
enc_dir.mkdir(exist_ok=True)
shutil.copy2(source_dir/"encoder"/"config.json",enc_dir/"config.json")
shutil.copy2(source_dir/"rl_agent_config.json",
             EXPORT_DIR/"rl_agent_config.json")
shutil.copy2(Path(exported_architecture.__file__),
             EXPORT_DIR/"all28_hf.py")
manifest={
 "architecture":"jev_cenn_all28_attention_only_frozen_ffn_v5",
 "base_model":SOURCE_MODEL,
 "attention_layers":manifest_layers,
 "all_28_attention_layers_replaced":True,
 "original_ffn_and_layernorm_fingerprint":after_ffn_sha,
 "original_ffns_preserved":True,
 "model_dtype":"float32",
 "model_parameters":sum(p.numel() for p in student_eval.parameters()),
 "source":"Forced best attention modules from V4 or fallback local distillation",
 "quality_acceptance_gate":False,
 "created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
 "transformers_version":"4.57.6"
}
(EXPORT_DIR/"all28_manifest.json").write_text(
    json.dumps(manifest,indent=2),encoding="utf-8")
requirements="""transformers==4.57.6
safetensors>=0.4
huggingface_hub>=0.25,<1
torch>=2.4
git+https://github.com/vtavakkoli/TinyCeNN-LM.git@eba220d8131dbdf06f748545a21831327e5ad37d
"""
(EXPORT_DIR/"requirements.txt").write_text(requirements)
readme="""---
library_name: pytorch
tags:
- cenn
- modernbert
- attention
- typed-decisions
- all-28
---

# Jev-CeNN all 28 attentions — original FFNs preserved

Complete trained typed-decision encoder model with all 28 attention modules
replaced by Jev-CeNN window/global-anchor alternatives. **FFNs, FFN norms,
attention norms, encoder depth (28), residual paths, and decision head are
preserved from the original source checkpoint.**

**Model is not directly compatible with** `transformers.AutoModel.from_pretrained`:
it uses `all28_hf.py` and `tinycenn_lm.standalone_decision`.

## Installation

```bash
pip install -r requirements.txt
```

## Load and run

```python
from huggingface_hub import snapshot_download
import sys
repo_id = "REPLACE_WITH_USERNAME/jev-cenn-all28-attention-frozen-ffn-v5"
root = snapshot_download(repo_id)  # authenticate first if private
sys.path.insert(0, root)
from all28_hf import load_all28
runtime = load_all28(root)
answer = runtime.decide(
    {"road_direction": "left", "speed": "7.0"},
    {"left": "turn left", "straight": "continue", "right": "turn right"},
)
print(answer)
```

## Model contents

- `model.safetensors`: complete pretrained+replacement parameters, NOT only
  attention patches
- `encoder/config.json`, `rl_agent_config.json`: full architecture settings
- `tokenizer/`: tokenization files
- `all28_manifest.json`: exact per-layer window/anchor/CeNN flags
- `all28_hf.py`: architecture and model loader
- `requirements.txt`: dependencies

The reported **94.8%** teacher agreement was measured on the author's prior
forced all-28 V4 research run. Reusing that exact checkpoint collection
should reproduce it under matched evaluation settings; newly trained
checkpoints may differ. This export does not enforce a quality/speed gate.
Dense masked SDPA does not guarantee an inference speed improvement.
Source checkpoint: `convaiinnovations/laya-typed-decisions`. Confirm
upstream licensing requirements before making the repository public.
"""
(EXPORT_DIR/"README.md").write_text(readme,encoding="utf-8")
print("Full-model safetensors:",round(model_file.stat().st_size/2**20,1),"MiB")
print("Export directory:",EXPORT_DIR)


In [ ]:
#@title 8. Standalone reload+logit round-trip: full model can actually run
import sys
sys.path.insert(0,str(EXPORT_DIR))
from all28_hf import build_all28_from_dir as verify_loaded_model
roundtrip=verify_loaded_model(EXPORT_DIR,device="cpu")
loaded=roundtrip.model
assert all(isinstance(x.attn,WindowCeNNAttention)
           for x in loaded.encoder.layers)
assert pretrained_sha_for_frozen_parts(loaded)==after_ffn_sha
# Semantic correctness not claimed by this smoke test. It verifies identical
# inference logits between saved model and fully reconstructed checkpoint.
sample=tokenizer('Road direction left, speed seven, decide now.',
                 return_tensors="pt",truncation=True,max_length=48)
positions=torch.tensor([[0,1]],dtype=torch.long)
smoke_input={
    "input_ids":sample["input_ids"],
    "attention_mask":sample["attention_mask"],
    "marker_pos":positions,
    "marker_mask":torch.ones(1,2,dtype=torch.bool),
    "qtype":torch.tensor([0],dtype=torch.long)}
with torch.inference_mode():
    native_logits,_=student_eval(**smoke_input)
    export_logits,_=loaded(**smoke_input)
max_abs=float((native_logits.float()-export_logits.float()).abs().max())
assert torch.isfinite(export_logits).all()
assert max_abs < 1e-4,f"Model serialization round-trip changed logits: {max_abs}"
print("PASS: FULL save/load logits match, max absolute error =",max_abs)
print("PASS: ALL 28 replaced attentions, original FFNs and norms unchanged.")
del loaded,roundtrip
gc.collect()


In [ ]:
#@title 9. AUTHENTICATE + CREATE PRIVATE HF MODEL REPO + UPLOAD FULL CHECKPOINT
if UPLOAD_TO_HF:
    token=get_token()
    if not token:
        print("Hugging Face authentication required; enter WRITE token in the secure login dialog.")
        login()
        token=get_token()
    if not token:
        raise RuntimeError("HF write token not found; use huggingface_hub.login()")
    api=HfApi(token=token)
    identity=api.whoami(token=token)
    owner=identity.get("name") or identity.get("fullname")
    if not owner:raise RuntimeError("Unable to detect Hugging Face namespace.")
    hf_repo=f"{owner}/{HF_REPO_NAME}"
    print("Publishing model to:",hf_repo,"private:",HF_PRIVATE)
    api.create_repo(repo_id=hf_repo,repo_type="model",
                    private=HF_PRIVATE,exist_ok=True)
    size=(EXPORT_DIR/"model.safetensors").stat().st_size
    if size>1024**3:
        # Large-folder resumable upload for very large models.
        api.upload_large_folder(repo_id=hf_repo,repo_type="model",
                                folder_path=str(EXPORT_DIR),num_workers=3)
    else:
        api.upload_folder(repo_id=hf_repo,repo_type="model",
                          folder_path=str(EXPORT_DIR),
                          commit_message="Publish complete all-28 forced attention model, original FFNs")
    remote_files=set(api.list_repo_files(repo_id=hf_repo,repo_type="model"))
    expected={"model.safetensors","all28_manifest.json","all28_hf.py",
              "README.md","encoder/config.json","rl_agent_config.json"}
    assert expected.issubset(remote_files),(
        "Upload may be incomplete; missing "+repr(sorted(expected-remote_files)))
    print("Hugging Face model published:",f"https://huggingface.co/{hf_repo}")
else:
    print("UPLOAD_TO_HF=False. Local completed model:",str(EXPORT_DIR))
